# Debug: Coordinate Mapping Analysis

Understanding why the white pixel distribution doesn't match Figure 4 from the paper.

## Key Question
How exactly should pixel coordinates (i,j) map to p-adic integers?

Current implementation uses hierarchical interleaving:
- Extract base-3 digits: i_digits, j_digits
- Interleave: j[0], i[0], j[1], i[1], j[2], i[2]

But maybe the paper uses a different mapping?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from padic.padic_embedding import embed_padic_cloud, get_paper_s
from scipy.ndimage import zoom
from keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("✓ Imports successful")

## Current Mapping Method (Hierarchical Interleaving)

In [ ]:
def coords_to_padic_CURRENT(i, j, p=3, l=6):
    """CURRENT: Hierarchical interleaving - j[0], i[0], j[1], i[1], j[2], i[2]"""
    i_digits = [i % 3, (i // 3) % 3, (i // 9) % 3]
    j_digits = [j % 3, (j // 3) % 3, (j // 9) % 3]
    padic_int = 0
    for k in range(l):
        digit = j_digits[k // 2] if k % 2 == 0 else i_digits[k // 2]
        padic_int += digit * (p ** k)
    return padic_int


def coords_to_padic_ALT1(i, j, p=3, l=6):
    """ALTERNATIVE 1: Reverse interleaving - i[0], j[0], i[1], j[1], i[2], j[2]"""
    i_digits = [i % 3, (i // 3) % 3, (i // 9) % 3]
    j_digits = [j % 3, (j // 3) % 3, (j // 9) % 3]
    padic_int = 0
    for k in range(l):
        digit = i_digits[k // 2] if k % 2 == 0 else j_digits[k // 2]
        padic_int += digit * (p ** k)
    return padic_int


def coords_to_padic_ALT2(i, j, p=3, l=6):
    """ALTERNATIVE 2: Concatenation - i then j"""
    i_digits = [i % 3, (i // 3) % 3, (i // 9) % 3]
    j_digits = [j % 3, (j // 3) % 3, (j // 9) % 3]
    padic_int = 0
    # i digits in lower positions
    for k in range(3):
        padic_int += i_digits[k] * (p ** k)
    # j digits in higher positions
    for k in range(3):
        padic_int += j_digits[k] * (p ** (k + 3))
    return padic_int


def coords_to_padic_ALT3(i, j, p=3, l=6):
    """ALTERNATIVE 3: Concatenation - j then i (reverse of ALT2)"""
    i_digits = [i % 3, (i // 3) % 3, (i // 9) % 3]
    j_digits = [j % 3, (j // 3) % 3, (j // 9) % 3]
    padic_int = 0
    # j digits in lower positions
    for k in range(3):
        padic_int += j_digits[k] * (p ** k)
    # i digits in higher positions
    for k in range(3):
        padic_int += i_digits[k] * (p ** (k + 3))
    return padic_int


print("Comparing different coordinate mapping approaches:")
print("\nExample: coordinate (i=5, j=13)")
print(f"  i=5 in base 3: [2, 1, 0] (2 + 1*3 + 0*9)")
print(f"  j=13 in base 3: [1, 1, 1] (1 + 1*3 + 1*9)")
print()
print(f"  CURRENT (j,i interleave): {coords_to_padic_CURRENT(5, 13)}")
print(f"  ALT1 (i,j interleave):    {coords_to_padic_ALT1(5, 13)}")
print(f"  ALT2 (i+j concat):        {coords_to_padic_ALT2(5, 13)}")
print(f"  ALT3 (j+i concat):        {coords_to_padic_ALT3(5, 13)}")

## Load Test MNIST Digit

In [ ]:
# Load MNIST digit 5 (same as in your recreation)
target_size = 27
img_raw = X_train[0]  # MNIST digit "5"
label = y_train[0]

# Resize and binarize
scale = target_size / img_raw.shape[0]
img_resized = zoom(img_raw.astype(np.float32), scale, order=1)
img_norm = img_resized / 255.0
img_binary = (img_norm > 0.5).astype(np.float32)

print(f"✓ Loaded MNIST digit {label}")
print(f"  Shape: {img_binary.shape}")
print(f"  Foreground pixels: {(img_binary > 0).sum()}")
print(f"  Foreground fraction: {(img_binary > 0).sum() / img_binary.size:.1%}")

# Visualize
plt.figure(figsize=(6, 6))
plt.imshow(img_binary, cmap='gray', origin='upper')
plt.title(f'MNIST Digit {label} (27×27)', fontsize=12, fontweight='bold')
plt.xlabel('Column (j)')
plt.ylabel('Row (i)')
plt.tight_layout()
plt.savefig('debug_digit.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Digit visualization saved")

## Compare Embedding Results from Different Methods

In [ ]:
p = 3
    l = 6
    m = 0
    s = get_paper_s("sierpinski_carpet", p=p, corrected=True)

    # Test each mapping method
    methods = [
        ("CURRENT (j,i interleave)", coords_to_padic_CURRENT),
        ("ALT1 (i,j interleave)", coords_to_padic_ALT1),
        ("ALT2 (i+j concat)", coords_to_padic_ALT2),
        ("ALT3 (j+i concat)", coords_to_padic_ALT3),
    ]

    results = {}

    for method_name, coords_func in methods:
        print(f"\nProcessing: {method_name}")
        
        # Create p-adic mapping using this method
        padic_mapping = np.zeros(target_size * target_size, dtype=np.int32)
        pixel_coords_i = np.zeros(target_size * target_size, dtype=np.int32)
        pixel_coords_j = np.zeros(target_size * target_size, dtype=np.int32)
        
        idx = 0
        for i in range(target_size):
            for j in range(target_size):
                padic_int = coords_func(i, j, p, l)
                padic_mapping[idx] = padic_int
                pixel_coords_i[idx] = i
                pixel_coords_j[idx] = j
                idx += 1
        
        # Embed in sierpinski space
        padic_points = embed_padic_cloud(padic_mapping, p=p, l=l, s=s, m=m)
        
        # Extract pixel values
        pixel_vals = np.zeros(target_size * target_size, dtype=np.float32)
        for idx in range(target_size * target_size):
            pixel_vals[idx] = img_binary[pixel_coords_i[idx], pixel_coords_j[idx]]
        
        results[method_name] = {
            'points': padic_points,
            'values': pixel_vals,
            'mapping': padic_mapping
        }
        
        # Verify properties
        fg_count = (pixel_vals > 0).sum()
        print(f"  ✓ Foreground points in sierpinski: {fg_count}")
        print(f"  ✓ Sierpinski X range: [{padic_points[:, 0].min():.3f}, {padic_points[:, 0].max():.3f}]")
        print(f"  ✓ Sierpinski Y range: [{padic_points[:, 1].min():.3f}, {padic_points[:, 1].max():.3f}]")

    print("\n✓ All methods computed")

## Visualize All Methods Side-by-Side

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    # Row 1: Original image + 3 methods
    ax = axes[0, 0]
    ax.imshow(img_binary, cmap='gray', origin='upper')
    ax.set_title('Original Image', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    for col, (method_name, _) in enumerate(methods):
        ax = axes[0, col + 1]
        points = results[method_name]['points']
        values = results[method_name]['values']
        
        ax.set_facecolor('#87CEEB')
        scatter = ax.scatter(points[:, 0], points[:, 1], c=values, cmap='binary',
                            s=30, alpha=0.8, edgecolors='none', vmin=0, vmax=1)
        ax.set_xlim([points[:, 0].min() - 0.15, points[:, 0].max() + 0.15])
        ax.set_ylim([points[:, 1].min() - 0.15, points[:, 1].max() + 0.15])
        ax.set_aspect('equal')
        ax.set_title(method_name, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.2)
    
    # Row 2: Zoomed in comparison of foreground patterns
    for col, (method_name, _) in enumerate(methods):
        ax = axes[1, col + 1]
        points = results[method_name]['points']
        values = results[method_name]['values']
        
        # Only plot foreground points for clarity
        fg_mask = values > 0
        fg_points = points[fg_mask]
        
        ax.scatter(fg_points[:, 0], fg_points[:, 1], c='black', s=50, alpha=0.8)
        ax.set_xlim([points[:, 0].min() - 0.15, points[:, 0].max() + 0.15])
        ax.set_ylim([points[:, 1].min() - 0.15, points[:, 1].max() + 0.15])
        ax.set_aspect('equal')
        ax.set_title(f'{method_name} (FG only)', fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.2)
    
    # Remove rightmost cell in row 2
    axes[1, 0].axis('off')
    
    plt.suptitle('Comparing Coordinate Mapping Methods\nWhich produces the correct distribution?',
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('debug_all_methods.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Comparison visualization saved")

## Analysis: Which Method Matches the Paper?

In [ ]:
print("="*70)
    print("ANALYSIS: WHICH MAPPING METHOD IS CORRECT?")
    print("="*70)
    
    print("""
    Looking at the visualizations above, we need to identify which mapping
    method produces a distribution of white pixels that matches Figure 4
    from the paper.
    
    Key observations to compare:
    1. Spatial clustering: Are white pixels grouped or scattered?
    2. Hierarchical structure: Do they form triangular/block patterns?
    3. Coverage: Do they fill certain regions of sierpinski space?
    4. Pattern recognition: Does the overall pattern look like the paper?
    
    The CURRENT method uses hierarchical interleaving: j[0], i[0], j[1], i[1], j[2], i[2]
    This was based on the MNIST notebook, but may not be the correct mapping.
    
    NEXT STEPS:
    1. Compare each method's output against the paper's Figure 4
    2. Identify which distribution matches
    3. Once identified, update the coords_to_padic function
    4. Regenerate all terrain analyses with the corrected mapping
    """)
    
    print("="*70)
    print("USER DECISION REQUIRED")
    print("="*70)
    print("""
    Please look at the four sierpinski embeddings above and compare them
    to Figure 4 from the paper. Which one has the correct white pixel
    distribution?
    
    Save your answer and I'll update the mapping function accordingly.
    """)